# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema available at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Dataset @id: {metadata.id}\n")


## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We will list all record sets, their fields, and columns as defined in the schema.

In [ ]:
# Enumerate available record sets, fields, and columns
record_sets = dataset.record_sets
all_record_sets = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field name: {field.name} (@id: {field.id}) [type: {field.data_type}]")
        if hasattr(field, 'column'):
            if isinstance(field.column, list):
                for col in field.column:
                    print(f"    - Column @id: {col.id}")
            else:
                print(f"    - Column @id: {field.column.id}")
    print("---\n")
    all_record_sets.append(rs.id)


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

In [ ]:
# Load every record set into a DataFrame using their @id
dfs = {}

for record_set_id in all_record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dfs[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for RecordSet @id: {record_set_id} -- {e}")

# Let's display the first RecordSet's head (if any data available)
if dfs:
    first_record_set_id = list(dfs.keys())[0]
    print(f"\nHead of DataFrame for RecordSet @id: {first_record_set_id}")
    dfs[first_record_set_id].head()
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter records, normalize numeric fields, categorize, group, and summarize.

We will select one record set and examine numeric fields. If available, filter and normalize.

In [ ]:
# Identify a record set with numeric fields
# Example: Let us check for a field called 'Age' (personalSensitiveInformation)
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

# Search for numeric fields
for rs in dataset.record_sets:
    for field in rs.fields:
        if field.data_type in ["Integer", "Float", "Number"] or field.name.lower() == "age":
            selected_record_set_id = rs.id
            numeric_field_id = field.id
            print(f"Selected RecordSet @id: {selected_record_set_id}")
            print(f"Numeric Field: {field.name} (@id: {numeric_field_id})")
            break
    if selected_record_set_id:
        break

# Try to find a reasonable group field (e.g., 'Sex')
if selected_record_set_id:
    for field in dataset.record_set(selected_record_set_id).fields:
        if field.data_type == "Text" and field.name.lower() in ["sex", "msi_status", "anatomical_location", "comorbidity"]:
            group_field_id = field.id
            print(f"Group Field: {field.name} (@id: {group_field_id})")
            break

# Proceed if DataFrame and numeric field are available
if selected_record_set_id and numeric_field_id and selected_record_set_id in dfs:
    df = dfs[selected_record_set_id]
    # Check presence of the column in DataFrame (may be key or readable alias)
    if numeric_field_id in df.columns:
        col = numeric_field_id
    elif "Age" in df.columns:
        col = "Age"
    else:
        col = df.select_dtypes(include=["int", "float"]).columns[0] if not df.empty else None

    # Filter entries where Age > 50
    threshold = 50
    print(f"\nFiltering {col} > {threshold}")
    filtered_df = df[df[col] > threshold]
    print(f"Filtered records:")
    pprint(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
    print(f"\nNormalized {col} for filtered records:")
    pprint(filtered_df[[col, f"{col}_normalized"]].head())

    # Group by group_field_id if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[col].mean().to_frame()
        print(f"\nMean {col} grouped by {group_field_id}:")
        pprint(grouped_df.head())
else:
    print("Suitable numeric field or data not found for EDA.")

## 5. Visualization

Visualize distributions and relationships using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example plot: Age distribution (if available)
if selected_record_set_id and selected_record_set_id in dfs:
    df = dfs[selected_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id], bins=10, kde=True)
        plt.xlabel("Age")
        plt.title("Distribution of Age")
        plt.show()

    # Example scatter: Age vs another numeric attribute, colored by group_field_id
    num_cols = df.select_dtypes(include=["int", "float"]).columns.tolist()
    if len(num_cols) > 1 and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=num_cols[0], y=num_cols[1], hue=group_field_id)
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.title(f"{num_cols[0]} vs {num_cols[1]} by {group_field_id}")
        plt.show()


## 6. Conclusion

This notebook demonstrated loading the FAIR^2 dataset via `mlcroissant`, extracting record sets, fields, and columns by their `@id`, and performing basic filtering, normalization, grouping, and visualization. Further domain-specific analyses can leverage the detailed structure provided by the Croissant schema for richer exploration.
